In [1]:
from pathlib import Path

import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras import callbacks

In [2]:

DATASET_DIR = Path("../data/archive/images")

TRAIN_DIR = DATASET_DIR.parent / "train"
VAL_DIR = DATASET_DIR.parent / "validation"
TEST_DIR = DATASET_DIR.parent / "test"


MODEL_DIR = Path(r"..\models")

MODEL_DIR.mkdir(exist_ok=True)

In [3]:
#parameters
IMG_SIZE = (224,224)

BATCH_SIZE = 32

SEED = 42

In [4]:
#loading dataset
train_ds = tf.keras.utils.image_dataset_from_directory(

    TRAIN_DIR,

    image_size=IMG_SIZE,

    batch_size=BATCH_SIZE,

    shuffle=True,

    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(

    VAL_DIR,

    image_size=IMG_SIZE,

    batch_size=BATCH_SIZE,

    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(

    TEST_DIR,

    image_size=IMG_SIZE,

    batch_size=BATCH_SIZE,

    shuffle=False
)

CLASS_NAMES = train_ds.class_names

NUM_CLASSES = len(CLASS_NAMES)

Found 10500 files belonging to 30 classes.
Found 2250 files belonging to 30 classes.
Found 2250 files belonging to 30 classes.


In [5]:
#optimizing dataset
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)

val_ds = val_ds.cache().prefetch(AUTOTUNE)

test_ds = test_ds.cache().prefetch(AUTOTUNE)

In [6]:
#data augmentation
data_augmentation = tf.keras.Sequential([

    layers.RandomFlip("horizontal"),

    layers.RandomRotation(0.1),

    layers.RandomZoom(0.1),

    layers.RandomTranslation(0.1,0.1)

])

In [7]:
#loading densenet121
base_model = tf.keras.applications.DenseNet121(

    include_top=False,

    weights="imagenet",

    input_shape=(224,224,3)
)

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [8]:
#freezing base model
base_model.trainable = False

In [9]:
#building the model
inputs = layers.Input(shape=(224,224,3))

x = data_augmentation(inputs)

x = tf.keras.applications.densenet.preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.4)(x)

x = layers.Dense(256, activation="relu")(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(

    NUM_CLASSES,

    activation="softmax"

)(x)

model = tf.keras.Model(inputs, outputs)

In [10]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ sequential (Sequential)              │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ true_divide (TrueDivide)             │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ add (Add)                            │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ true_divide_1 (TrueDivide)           │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ densenet121 (Functional)             │ (None, 7, 7, 1024)          │       7,037,504 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1024)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 1024)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         262,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 30)                  │           7,710 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 7,307,614 (27.88 MB)

 Trainable params: 270,110 (1.03 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [11]:
model.compile(

    optimizer=tf.keras.optimizers.Adam(

        learning_rate=1e-3

    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)

In [12]:
early_stop = callbacks.EarlyStopping(

    monitor="val_loss",

    patience=5,

    restore_best_weights=True
)

reduce_lr = callbacks.ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.3,

    patience=2,

    verbose=1
)

checkpoint = callbacks.ModelCheckpoint(

    MODEL_DIR/"densenet_best.keras",

    monitor="val_accuracy",

    save_best_only=True
)

In [14]:
#initial training
history = model.fit(

    train_ds,

    validation_data=val_ds,

    epochs=10,

    callbacks=[

        early_stop,

        reduce_lr,

        checkpoint

    ]
)

Epoch 1/10
329/329 ━━━━━━━━━━━━━━━━━━━━ 798s 2s/step - accuracy: 0.3977 - loss: 2.1137 - val_accuracy: 0.7111 - val_loss: 0.9525 - learning_rate: 0.0010
Epoch 2/10
329/329 ━━━━━━━━━━━━━━━━━━━━ 767s 2s/step - accuracy: 0.5708 - loss: 1.4204 - val_accuracy: 0.7347 - val_loss: 0.8068 - learning_rate: 0.0010
Epoch 3/10
329/329 ━━━━━━━━━━━━━━━━━━━━ 769s 2s/step - accuracy: 0.6066 - loss: 1.2708 - val_accuracy: 0.7524 - val_loss: 0.7541 - learning_rate: 0.0010
Epoch 4/10
329/329 ━━━━━━━━━━━━━━━━━━━━ 1000s 3s/step - accuracy: 0.6228 - loss: 1.1989 - val_accuracy: 0.7640 - val_loss: 0.7050 - learning_rate: 0.0010
Epoch 5/10
329/329 ━━━━━━━━━━━━━━━━━━━━ 1269s 4s/step - accuracy: 0.6371 - loss: 1.1705 - val_accuracy: 0.7764 - val_loss: 0.6969 - learning_rate: 0.0010
Epoch 6/10
329/329 ━━━━━━━━━━━━━━━━━━━━ 1286s 4s/step - accuracy: 0.6530 - loss: 1.1121 - val_accuracy: 0.7782 - val_loss: 0.6746 - learning_rate: 0.0010
Epoch 7/10
329/329 ━━━━━━━━━━━━━━━━━━━━ 1013s 3s/step - accuracy: 0.6515 - loss

In [15]:
#fine tuning
base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False

In [16]:
#recompile
model.compile(

    optimizer=tf.keras.optimizers.Adam(

        learning_rate=1e-5

    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)

In [17]:
#Continue training
history_finetune = model.fit(

    train_ds,

    validation_data=val_ds,

    epochs=15,

    callbacks=[

        early_stop,

        reduce_lr,

        checkpoint

    ]
)

Epoch 1/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 1276s 4s/step - accuracy: 0.6218 - loss: 1.2879 - val_accuracy: 0.7929 - val_loss: 0.6590 - learning_rate: 1.0000e-05
Epoch 2/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 832s 3s/step - accuracy: 0.6504 - loss: 1.1562 - val_accuracy: 0.7933 - val_loss: 0.6259 - learning_rate: 1.0000e-05
Epoch 3/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 854s 3s/step - accuracy: 0.6656 - loss: 1.0909 - val_accuracy: 0.7960 - val_loss: 0.6096 - learning_rate: 1.0000e-05
Epoch 4/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 833s 3s/step - accuracy: 0.6746 - loss: 1.0680 - val_accuracy: 0.8022 - val_loss: 0.6011 - learning_rate: 1.0000e-05
Epoch 5/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 847s 3s/step - accuracy: 0.6794 - loss: 1.0348 - val_accuracy: 0.8071 - val_loss: 0.5938 - learning_rate: 1.0000e-05
Epoch 6/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 831s 3s/step - accuracy: 0.6850 - loss: 1.0018 - val_accuracy: 0.8062 - val_loss: 0.5880 - learning_rate: 1.0000e-05
Epoch 7/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 909s 3s/step - ac

In [ ]:
# ###history_finetune2 = model.fit(

#     train_ds,

#     validation_data=val_ds,

#     initial_epoch=15,

#     epochs=20,

#     callbacks=[
#         early_stop,
#         reduce_lr,
#         checkpoint
#     ]
# )##

In [18]:
#saving final model
model.save(

    MODEL_DIR/"densenet_final.keras"
)